### This code demonstrates the problem that LangChain's team faced

**Simple idea:**  
We have separate pieces such as a PromptTemplate and an LLM.  
If we connect them manually every time, the code becomes repetitive.

The demo below creates a simple **Chain** that connects these pieces for us.

In [ ]:
# numpy is imported here, although it is not used in this demo.
import numpy

In [2]:
import random

# This is a fake/mock LLM.
# We use it only to understand how LangChain works internally.
class NakliLLM:
    def __init__(self):
        print("LLM Created..")

    def predict(self, prompt):
        # A real LLM would generate a response from the prompt.
        # Here we simply return one random response.
        response_list = [
            "Delhi is the capital of India",
            "Ms. Cailin is my favourite character",
            "Satoru Gojo is the Honored One"
        ]

        return {"response": random.choice(response_list)}

In [3]:
# Create our fake LLM object.
llm = NakliLLM()

LLM Created..


In [7]:
# Send a question to the LLM.
# In a real application, the LLM would generate an answer.
llm.predict("What is my favourite character?")

{'response': 'Ms. Cailin is my favourite character'}

In [8]:
class NakliPromptTemplate:
    def __init__(self, template, input_variables):
        # Store the prompt template.
        self.template = template

        # Store the names of the values that we will provide later.
        self.input_variables = input_variables

    def format(self, input_dict):
        # Replace placeholders such as {topic} and {mode}
        # with the values provided by the user.
        return self.template.format(**input_dict)

In [14]:
# Create a reusable prompt template.
# {topic} and {mode} are placeholders.
prompt = NakliPromptTemplate(
    template = "Explain {topic} in easy but in {mode} mode",
    input_variables = ["topic", "mode"]
)

In [16]:
# Fill the placeholders with actual values.
prompt.format({"mode": "technical", "topic": "DL"})

'Explain DL in easy but in technical mode'

## Combining the NakliLLM & NakliPromptTemplate to create a Chain

**Why do we need a Chain?**

Without a Chain, we have to manually:
1. Format the prompt.
2. Send the formatted prompt to the LLM.
3. Extract the response.

A Chain puts these steps together so we can call one method and let it handle the flow.

In [17]:
# Create the LLM again for the next example.
llm = NakliLLM()

LLM Created..


In [18]:
# Here we try to send the prompt object directly to the LLM.
# This shows that we still have to manually connect
# the PromptTemplate and LLM together.
llm.predict(prompt = prompt)

{'response': 'Delhi is the capital of India'}

In [19]:
class NakliLLMChain:

    def __init__(self, llm, prompt):
        # Store both components inside the Chain.
        self.llm = llm
        self.prompt = prompt

    def run(self, input_dict):
        # Step 1: Fill the prompt placeholders.
        final_prompt = self.prompt.format(input_dict)

        # Step 2: Send the final prompt to the LLM.
        result = self.llm.predict(final_prompt)

        # Step 3: Return only the actual response.
        return result["response"]

In [20]:
# Create another reusable prompt template.
template = NakliPromptTemplate(
    template= "Explain the anime {name} in details.",
    input_variables = ["name"]
)

In [21]:
# Create the fake LLM that the Chain will use.
llm = NakliLLM()

LLM Created..


In [25]:
# Connect the prompt and LLM inside our Chain.
llm_chain = NakliLLMChain(llm = llm, prompt = template)

# The Chain now does the work for us:
# 1. Fill the prompt
# 2. Send it to the LLM
# 3. Return the response
result = llm_chain.run({"name": "Battle Through The Heavens"})
result

'Ms. Cailin is my favourite character'